create table of highest correlations

In [ ]:
import itertools
from pathlib import Path
import typing

import pandas as pd

from plot_corr_assoc import create_corr_assoc_matrix
from utils import load_data

# Hardcoded paths, change to apply to new data
data_path = Path("data/preprocessed/esa_2018_2021.pkl")
vars_info_path = Path("data/preprocessed/esa_vars_info.pkl")

image_folder = "outputs/corr_assoc_plots"
plot_save_name_template = "corr_assoc_plot_{group}.png"

allow_overwrite = True

# Read in data
data: pd.DataFrame = load_data(data_path)
vars_info: pd.DataFrame = load_data(vars_info_path)

# Get variable groups
var_groups = {
    group: vars_info.loc[vars_info["group"] == group, "name"] for group in vars_info["group"].cat.categories
}
var_groups = typing.cast(dict[str, list[str]], var_groups)

# Visualize correlation matrices
treat_cat_as_continuous: list[str] = vars_info.loc[vars_info["type"].isin(["binary", "ordinal"]), "name"].tolist()

corr_assoc_matrix, p_value_matrix = create_corr_assoc_matrix(data, treat_cat_as_continuous)
table = (
    pd.DataFrame(
        [
            [vari, varj, corr_assoc, p_value_matrix.loc[vari, varj]]
            for vari, varj in itertools.combinations(vars_info["name"], r=2)
            if (corr_assoc := corr_assoc_matrix.loc[vari, varj]) > 0.9
        ]
    )
    .set_axis(["vari", "varj", "corr_assoc", "p_value"], axis=1)
    .sort_values("corr_assoc", ascending=False)
)
table

search for variable information

In [ ]:
vars_info[vars_info["name"].str.contains("hnb")]